In [10]:
# ==================================================================================
# 🚀 AGENT IA V24 : STRUCTURE ROBUSTE (SANS CACHE-MISÈRE)
# ==================================================================================

# --- 1. INSTALLATION ---
import os
print("⏳ Installation des librairies... (1 minute)")
!pip install -q odfpy python-pptx python-docx docx2txt reportlab pdf2image openpyxl xlsxwriter pymupdf4llm "smolagents[litellm]" langchain langchain-community langchain-google-genai langchain-text-splitters duckduckgo-search faiss-cpu
!apt-get install -y poppler-utils > /dev/null 2>&1
print("✅ Installation terminée !")

⏳ Installation des librairies... (1 minute)
✅ Installation terminée !


In [4]:
# --- 2. CONFIGURATION & IMPORTS ---
import time
import pandas as pd
import os
import zipfile
import xml.etree.ElementTree as ET
import google.generativeai as genai
import subprocess
import io, warnings, csv
import docx2txt
import pymupdf4llm
import openpyxl
import xlsxwriter

from smolagents import CodeAgent, LiteLLMModel, tool, DuckDuckGoSearchTool
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import MarkdownTextSplitter
from openpyxl.styles import Font, PatternFill, Alignment
from pptx import Presentation
from pptx.util import Inches, Pt
from docx import Document
from docx.shared import Inches as DocxInches
from PIL import Image
from reportlab.pdfgen import canvas
try: from pdf2image import convert_from_path
except: pass

warnings.filterwarnings("ignore")

In [ ]:
# --- TA CLÉ API ---
if 'CLE_GOOGLE_VISION' not in globals():
    # REMPLACE CECI PAR TA CLÉ SI ELLE N'EST PAS DÉJÀ EN MÉMOIRE
    CLE_GOOGLE_VISION = "MA_CLE_SECURISEE"

genai.configure(api_key=CLE_GOOGLE_VISION)
os.environ["GOOGLE_API_KEY"] = CLE_GOOGLE_VISION

In [6]:
# --- 3. LE CERVEAU & LA MÉMOIRE (CRITIQUE) ---
print("🔌 Connexion au Cerveau (Gemini)...")
# PAS DE TRY/EXCEPT ICI : Si ça plante, on veut savoir pourquoi TOUT DE SUITE.
model_agent = LiteLLMModel(model_id="gemini/gemini-2.5-flash", api_key=CLE_GOOGLE_VISION, temperature=0.1)
print("✅ Cerveau Connecté.")

print("📥 Chargement de la Mémoire (Google 004)...")
# On tente l'embedding 004. S'il échoue, on arrête tout pour ne pas avoir de résultats vides.
try:
    embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
    embeddings.embed_query("test") # Test de vie
    print("✅ Mémoire Active (Google 004).")
except Exception as e:
    print(f"\n🛑 ERREUR EMBEDDING FATALE : {e}")
    print("Ton code va s'arrêter car sans mémoire, l'agent est aveugle.")
    raise e # On force l'arrêt ici

🔌 Connexion au Cerveau (Gemini)...
✅ Cerveau Connecté.
📥 Chargement de la Mémoire (Google 004)...
✅ Mémoire Active (Google 004).


In [7]:
# ==============================================================================
# 4. MOTEURS DE LECTURE
# ==============================================================================

def appel_gemini_securise(prompt, image=None):
    """Appel Gemini Vision sécurisé"""
    max_retries = 3
    model = genai.GenerativeModel("gemini-2.0-flash")
    for i in range(max_retries):
        try:
            if image: return model.generate_content([prompt, image]).text
            else: return model.generate_content(prompt).text
        except Exception as e:
            if "429" in str(e): time.sleep(5)
            else: return f"Erreur Vision : {e}"
    return "Erreur Quota."

def lire_xlsx_xml_natif(chemin_fichier):
    """Lecture Excel via XML"""
    try:
        data = []
        with zipfile.ZipFile(chemin_fichier, 'r') as z:
            shared_strings = []
            if 'xl/sharedStrings.xml' in z.namelist():
                with z.open('xl/sharedStrings.xml') as f:
                    tree = ET.parse(f); root = tree.getroot()
                    for si in root.findall('.//{http://schemas.openxmlformats.org/spreadsheetml/2006/main}si'):
                        texts = [t.text for t in si.findall('.//{http://schemas.openxmlformats.org/spreadsheetml/2006/main}t') if t.text]
                        shared_strings.append("".join(texts))
            sheets = [n for n in z.namelist() if n.startswith('xl/worksheets/sheet') and n.endswith('.xml')]
            for sheet_file in sheets:
                data.append(f"\n--- FEUILLE XML : {sheet_file} ---\n")
                with z.open(sheet_file) as f:
                    tree = ET.parse(f); root = tree.getroot()
                    for row in root.findall('.//{http://schemas.openxmlformats.org/spreadsheetml/2006/main}row'):
                        row_data = []
                        for cell in row.findall('.//{http://schemas.openxmlformats.org/spreadsheetml/2006/main}c'):
                            cell_type = cell.get('t')
                            val_tag = cell.find('{http://schemas.openxmlformats.org/spreadsheetml/2006/main}v')
                            val = ""
                            if val_tag is not None and val_tag.text:
                                if cell_type == 's':
                                    try: val = shared_strings[int(val_tag.text)]
                                    except: val = val_tag.text
                                else: val = val_tag.text
                            row_data.append(val)
                        if any(row_data): data.append(" | ".join(row_data))
        return "\n".join(data)
    except: return pd.read_excel(chemin_fichier).fillna("").to_markdown(index=False)

def lire_xml_natif_generique(chemin_fichier, type_doc):
    """Lecture LibreOffice via XML"""
    try:
        data = []
        with zipfile.ZipFile(chemin_fichier, 'r') as z:
            with z.open('content.xml') as f:
                tree = ET.parse(f); root = tree.getroot()
                ns = {'table': 'urn:oasis:names:tc:opendocument:xmlns:table:1.0',
                      'text': 'urn:oasis:names:tc:opendocument:xmlns:text:1.0',
                      'draw': 'urn:oasis:names:tc:opendocument:xmlns:drawing:1.0'}
                if type_doc == 'spreadsheet':
                    for table in root.findall('.//table:table', ns):
                        t_name = table.get(f"{{{ns['table']}}}name"); data.append(f"\n--- FEUILLE : {t_name} ---\n")
                        for row in table.findall('.//table:table-row', ns):
                            row_data = []
                            for cell in row.findall('.//table:table-cell', ns):
                                repeat = cell.get(f"{{{ns['table']}}}number-columns-repeated")
                                texts = [t.text for t in cell.findall('.//text:p', ns) if t.text]
                                val = " ".join(texts).strip(); row_data.append(val if val else "")
                                if repeat:
                                    try:
                                        for _ in range(int(repeat)-1): row_data.append("")
                                    except: pass
                            if any(row_data): data.append(" | ".join(row_data))
                elif type_doc == 'presentation':
                    for i, slide in enumerate(root.findall('.//draw:page', ns)):
                        data.append(f"\n--- Slide {i+1} ---")
                        texts = [p.text for p in slide.findall('.//text:p', ns) if p.text]
                        data.append("\n".join(texts))
                elif type_doc == 'text':
                    data.append(f"[ODT: {os.path.basename(chemin_fichier)}]")
                    for p in root.findall('.//text:p', ns):
                        if p.text: data.append(p.text)
        return "\n".join(data)
    except Exception as e: return f"Erreur XML ({type_doc}): {e}"

def convertir_tout_document(chemin_fichier):
    """Dispatcher Universel"""
    if not os.path.exists(chemin_fichier): return ""
    ext = os.path.splitext(chemin_fichier)[1].lower()
    texte = ""
    print(f"📂 Lecture ({ext}) : {os.path.basename(chemin_fichier)}")
    try:
        if ext == ".csv":
            try: texte = pd.read_csv(chemin_fichier).fillna("").to_markdown(index=False)
            except: pass
        elif ext in [".xlsx", ".xls"]: texte = lire_xlsx_xml_natif(chemin_fichier)
        elif ext == ".ods": texte = lire_xml_natif_generique(chemin_fichier, 'spreadsheet')
        elif ext == ".odp": texte = lire_xml_natif_generique(chemin_fichier, 'presentation')
        elif ext == ".odt": texte = lire_xml_natif_generique(chemin_fichier, 'text')
        elif ext == ".pptx":
            try:
                prs = Presentation(chemin_fichier); texte += f"\n[PPTX: {os.path.basename(chemin_fichier)}]\n"
                for i, slide in enumerate(prs.slides):
                    texte += f"\n--- Slide {i+1} ---\n"
                    for shape in slide.shapes:
                        if hasattr(shape, "text"): texte += shape.text + "\n"
            except Exception as e: texte = f"Erreur PPTX: {e}"
        elif ext == ".docx": texte = docx2txt.process(chemin_fichier)
        # LE POINT CRUCIAL : PyMuPDF pour les PDF
        elif ext == ".pdf": texte = pymupdf4llm.to_markdown(chemin_fichier)
        elif ext in [".jpg", ".png", ".jpeg"]:
            img = Image.open(chemin_fichier)
            texte = appel_gemini_securise("Décris cette image en détail.", img)
        elif ext == ".txt":
            with open(chemin_fichier, 'r', encoding='utf-8', errors='ignore') as f: texte = f.read()
    except Exception as e: return f"Erreur lecture globale: {e}"
    return texte

In [8]:
# --- FONCTION RAG ---
vectorstore_global = None
def initialiser_rag(fichiers):
    global vectorstore_global
    text_data = ""
    print(f"📚 Indexation de {len(fichiers)} fichiers...")
    for f in fichiers:
        content = convertir_tout_document(f)
        text_data += content + "\n\n"

    if not text_data.strip():
        print("⚠️ Aucun texte extrait.")
        return

    chunks = MarkdownTextSplitter(chunk_size=1000, chunk_overlap=200).split_text(text_data)

    try:
        vectorstore_global = FAISS.from_texts(chunks, embeddings)
        print(f"✅ Mémoire chargée avec succès ({len(chunks)} fragments).")
    except Exception as e:
        print(f"❌ Erreur Vectorisation : {e}")

In [9]:
# ==============================================================================
# 5. OUTILS D'ACTION (DOCSTRINGS COMPLÈTES ET VERBEUSES)
# ==============================================================================

@tool
def outil_rag(question: str) -> str:
    """
    Cherche une information TEXTUELLE dans les documents chargés (PDF, Excel, Word...).
    Utilise cet outil en PREMIER pour trouver des chiffres, des noms ou des faits.
    Attention : Cet outil ne "voit" pas les images scannées.

    Args:
        question: La question précise à poser à la base de documents (ex: "Quel est le chiffre d'affaires ?").
    """
    global vectorstore_global
    if vectorstore_global is None: return "Aucun document chargé."
    try:
        docs = vectorstore_global.similarity_search(question, k=10)
        return "\n---\n".join([d.page_content for d in docs])
    except: return "Erreur RAG"

@tool
def vision_page_pdf(chemin_pdf: str, numero_page: str, question: str) -> str:
    """
    Analyse UNIQUEMENT une page précise d'un PDF avec la Vision Artificielle.
    Utilise cet outil SI ET SEULEMENT SI 'outil_rag' n'a pas trouvé l'info car elle est dans un tableau complexe, un scan ou un graphique.

    Args:
        chemin_pdf: Le chemin du fichier PDF (ex: 'document.pdf').
        numero_page: Le numéro de la page à analyser (ex: '4'). ATTENTION: commence à 1.
        question: Ce qu'il faut chercher ou transcrire sur cette page (ex: 'Transcris le tableau des passifs').
    """
    try:
        if not os.path.exists(chemin_pdf): return "Fichier introuvable."
        page_idx = int(numero_page) - 1
        images = convert_from_path(chemin_pdf, first_page=page_idx+1, last_page=page_idx+1)
        if not images: return "Page introuvable."
        return appel_gemini_securise(f"{question}", images[0])
    except Exception as e: return f"Erreur Sniper Vision: {e}"

@tool
def chirurgien_excel(nom_fichier: str, cellule: str, valeur: str, action: str = "ECRIRE") -> str:
    """
    Outil pour créer ou modifier un fichier Excel (.xlsx).
    Permet d'écrire, de fusionner et de mettre en forme.

    Args:
        nom_fichier: Le chemin du fichier Excel cible (ex: 'audit.xlsx').
        cellule: La cellule cible (ex: 'B2') ou la plage pour une fusion (ex: 'A1:C1').
        valeur: La valeur textuelle ou numérique à insérer dans la cellule.
        action: L'action à réaliser. Les valeurs acceptées sont :
                - 'ECRIRE' : Écrit simplement la valeur.
                - 'FUSION' : Fusionne les cellules et écrit la valeur.
                - 'GRAS' : Met la cellule en gras.
                - 'ROUGE' : Met le texte en rouge.
                - 'FOND_JAUNE' : Met un fond jaune.
                - 'TITRE_STYLYSE' : Applique un style de titre (Fond bleu, texte blanc, gras, centré).
    """
    if not os.path.exists(nom_fichier): wb = openpyxl.Workbook(); wb.save(nom_fichier)
    try:
        wb = openpyxl.load_workbook(nom_fichier); ws = wb.active
        val_clean = valeur
        try: val_clean = float(valeur.replace(' ','').replace('€','').replace(',','.'))
        except: pass

        if action == "ECRIRE": ws[cellule] = val_clean
        elif action == "FUSION":
            ws.merge_cells(cellule); top_left = cellule.split(':')[0]
            ws[top_left] = valeur
            ws[top_left].alignment = Alignment(horizontal='center', vertical='center')

        c = ws[cellule.split(':')[0]]
        if action == "GRAS": c.font = Font(bold=True)
        elif action == "ROUGE": c.font = Font(color="FF0000")
        elif action == "FOND_JAUNE": c.fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
        elif action == "TITRE_STYLYSE":
            c.font = Font(bold=True, color="FFFFFF", size=12)
            c.fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
            c.alignment = Alignment(horizontal='center', vertical='center')
            if valeur: c.value = valeur

        if valeur and action in ["GRAS", "ROUGE", "FOND_JAUNE"]: ws[cellule] = val_clean
        wb.save(nom_fichier)
        return f"Succès Excel : {action} sur {cellule}."
    except Exception as e: return f"Erreur Excel: {e}"

@tool
def chirurgien_word_ppt(nom_fichier: str, texte_ancrage: str, texte_ajout: str, action: str = "AJOUTER_CONTENU", chemin_image: str = None, numero_slide: str = "DERNIERE") -> str:
    """
    Outil polyvalent pour modifier Word (.docx) et PowerPoint (.pptx).

    Args:
        nom_fichier: Chemin du fichier cible.
        texte_ancrage: (Pour Word et PPT-Remplacer) Le texte repère existant qu'on cherche.
        texte_ajout: Le nouveau texte à ajouter.
        action: L'action à effectuer :
                - 'AJOUTER_CONTENU' : Ajoute le texte après l'ancrage (Word) ou dans une nouvelle zone (PPT).
                - 'REMPLACER' : Remplace 'texte_ancrage' par 'texte_ajout'.
                - 'SUPPRIMER_DIAPO' : Supprime une diapo (PPT uniquement).
        chemin_image: (Optionnel) Chemin d'une image à insérer.
        numero_slide: (Pour PPT) Numéro de la diapo cible (ex: '1', '2' ou 'DERNIERE').
    """
    try:
        ext = os.path.splitext(nom_fichier)[1].lower()
        if ext == '.docx':
            if not os.path.exists(nom_fichier): Document().save(nom_fichier)
            doc = Document(nom_fichier)
            if action == "REMPLACER":
                for p in doc.paragraphs:
                    if texte_ancrage and texte_ancrage in p.text:
                        p.text = p.text.replace(texte_ancrage, texte_ajout)
            elif action == "AJOUTER_CONTENU":
                doc.add_paragraph(texte_ajout)
                if chemin_image: doc.add_picture(chemin_image, width=DocxInches(4))
            doc.save(nom_fichier)
            return "Succès Word."
        elif ext == '.pptx':
            if not os.path.exists(nom_fichier): Presentation().save(nom_fichier)
            prs = Presentation(nom_fichier)
            idx = len(prs.slides)-1 if numero_slide=="DERNIERE" else int(numero_slide)-1
            if action == "SUPPRIMER_DIAPO":
                xml_slides = prs.slides._sldIdLst; slides = list(xml_slides); xml_slides.remove(slides[idx])
                prs.save(nom_fichier); return "Diapo supprimée."
            slide = prs.slides[idx]
            max_top = 0
            for sh in slide.shapes: max_top = max(max_top, sh.top + sh.height)
            if max_top > Inches(6.5):
                slide = prs.slides.add_slide(prs.slide_layouts[5]); max_top = Inches(1)
            if texte_ajout:
                tb = slide.shapes.add_textbox(Inches(1), max_top + Inches(0.5), Inches(8), Inches(1))
                tb.text = texte_ajout
                max_top += Inches(1)
            if chemin_image:
                slide.shapes.add_picture(chemin_image, Inches(1), max_top + Inches(0.5), height=Inches(3))
            prs.save(nom_fichier)
            return "Succès PPT."
    except Exception as e: return f"Erreur Office: {e}"

@tool
def generateur_pdf_natif(nom_fichier: str, texte: str) -> str:
    """
    Crée un fichier PDF simple à partir de texte.
    Args:
        nom_fichier: Le nom du fichier PDF à créer (ex: 'rapport.pdf').
        texte: Le contenu textuel à écrire dans le PDF.
    """
    try:
        c = canvas.Canvas(nom_fichier, pagesize=A4); y=800; c.setFont("Helvetica", 12)
        for line in texte.split('\n'):
            c.drawString(50, y, line); y-=20;
            if y<50: c.showPage(); y=800
        c.save(); return "PDF Créé."
    except: return "Erreur PDF"

@tool
def convertisseur_universel(chemin_source: str, format_cible: str) -> str:
    """
    Convertit DOCX ou PPTX en PDF, ODT ou ODP.
    Args:
        chemin_source: Le fichier à convertir.
        format_cible: Le format de sortie ('pdf', 'odt', 'odp').
    """
    try:
        subprocess.run(['libreoffice', '--headless', '--convert-to', format_cible, chemin_source, '--outdir', '.'], check=True, stdout=subprocess.DEVNULL)
        try: os.remove(chemin_source)
        except: pass
        return "Conversion OK."
    except: return "Erreur Convert"

@tool
def deverrouilleur_pdf(chemin_pdf: str) -> str:
    """
    Convertit un PDF en DOCX pour pouvoir le modifier.
    Args:
        chemin_pdf: Le fichier PDF source.
    """
    try:
        subprocess.run(['libreoffice', '--headless', '--convert-to', 'docx', chemin_pdf, '--outdir', '.'], check=True, stdout=subprocess.DEVNULL)
        return os.path.splitext(chemin_pdf)[0]+".docx"
    except: return "Erreur Unlock"

@tool
def editeur_texte_csv(nom_fichier: str, contenu: str, mode: str = "AJOUTER_FIN") -> str:
    """
    Crée ou modifie un fichier texte (.txt) ou CSV (.csv).
    Args:
        nom_fichier: Le nom du fichier.
        contenu: Le texte à écrire.
        mode: 'AJOUTER_FIN' (ajoute à la fin) ou 'ECRASER' (remplace tout).
    """
    try:
        m = 'w' if mode=="ECRASER" else 'a'
        with open(nom_fichier, m, encoding='utf-8') as f: f.write("\n"+contenu)
        return "Fichier Texte Modifié."
    except: return "Erreur TXT"

@tool
def outil_vision(chemin_image: str, question: str) -> str:
    """
    Analyse une image simple (JPG, PNG).
    Args:
        chemin_image: Le chemin de l'image.
        question: La question à poser sur l'image.
    """
    try: return appel_gemini_securise(f"{question}", Image.open(chemin_image))
    except: return "Erreur Vision"

@tool
def web_search(query: str) -> str:
    """
    Recherche des informations sur Internet.
    Args:
        query: Les mots-clés de la recherche.
    """
    try: return DuckDuckGoSearchTool().run(query)
    except: return "Erreur Web"

In [10]:
# ASSEMBLAGE FINAL
liste_outils = [outil_rag, vision_page_pdf, chirurgien_excel, chirurgien_word_ppt, generateur_pdf_natif, convertisseur_universel, deverrouilleur_pdf, editeur_texte_csv, outil_vision, web_search]
imports = ["os", "pandas", "zipfile", "openpyxl", "pptx", "docx", "subprocess", "reportlab", "PIL", "csv", "pdf2image"]

agent = CodeAgent(
    model=model_agent, # Le modèle est bien défini plus haut
    tools=liste_outils,
    additional_authorized_imports=imports,
    max_steps=20
)

print("✅ AGENT V25 OPÉRATIONNEL (DOCSTRINGS RÉPARÉES).")

✅ AGENT V25 OPÉRATIONNEL (DOCSTRINGS RÉPARÉES).


In [11]:
# Mission Audit
# 1. On charge le fichier généré
fichiers_mission = ['rapport_financier_fictif.pdf']
initialiser_rag(fichiers_mission)

# 2. La Mission
mission_audit = """
Agis comme un Auditeur Financier.
Analyse le document PDF 'rapport_financier_fictif.pdf'.

TACHE 1 : Extraction
Trouve :
- Le Chiffre d'Affaires.
- Le Résultat Net.
- Le nom du PDG.
- Le Ratio de Solvabilité (caché dans le texte).

TACHE 2 : Création Excel
Crée un fichier 'Audit_TechNova.xlsx'.
1. Crée les colonnes 'Indicateur' et 'Valeur'.
2. Remplis avec les données trouvées.
3. Mets la ligne d'en-tête (Ligne 1) en GRAS, ROUGE et CENTRÉ.
"""

print(f"🎯 MISSION AUDIT : {mission_audit}")
try:
    res = agent.run(mission_audit)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

📚 Indexation de 1 fichiers...
📂 Lecture (.pdf) : rapport_financier_fictif.pdf
✅ Mémoire chargée avec succès (3 fragments).
🎯 MISSION AUDIT : 
Agis comme un Auditeur Financier.
Analyse le document PDF 'rapport_financier_fictif.pdf'.

TACHE 1 : Extraction
Trouve :
- Le Chiffre d'Affaires.
- Le Résultat Net.
- Le nom du PDG.
- Le Ratio de Solvabilité (caché dans le texte).

TACHE 2 : Création Excel
Crée un fichier 'Audit_TechNova.xlsx'.
1. Crée les colonnes 'Indicateur' et 'Valeur'.
2. Remplis avec les données trouvées.
3. Mets la ligne d'en-tête (Ligne 1) en GRAS, ROUGE et CENTRÉ.



╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Agis comme un Auditeur Financier.                                                                               │
│ Analyse le document PDF 'rapport_financier_fictif.pdf'.                                                         │
│                                                                                                                 │
│ TACHE 1 : Extraction                                                                                            │
│ Trouve :                                                                                                        │
│ - Le Chiffre d'Affaires.                                                                                        │
│ - Le Résultat Net.                                                                                              │
│ - Le nom du PDG.                                                                                                │
│ - Le Ratio de Solvabilité (caché dans le texte).                                                                │
│                                                                                                                 │
│ TACHE 2 : Création Excel                                                                                        │
│ Crée un fichier 'Audit_TechNova.xlsx'.                                                                          │
│ 1. Crée les colonnes 'Indicateur' et 'Valeur'.                                                                  │
│ 2. Remplis avec les données trouvées.                                                                           │
│ 3. Mets la ligne d'en-tête (Ligne 1) en GRAS, ROUGE et CENTRÉ.                                                  │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-2.5-flash ────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  chiffre_affaires = outil_rag(question="Quel est le Chiffre d'Affaires ?")                                        
  print(f"Chiffre d'Affaires: {chiffre_affaires}")                                                                 
                                                                                                                   
  resultat_net = outil_rag(question="Quel est le Résultat Net ?")                                                  
  print(f"Résultat Net: {resultat_net}")                                                                           
                                                                                                                   
  nom_pdg = outil_rag(question="Quel est le nom du PDG ?")                                                         
  print(f"Nom du PDG: {nom_pdg}")                                                                                  
                                                                                                                   
  ratio_solvabilite = outil_rag(question="Quel est le Ratio de Solvabilité ?")                                     
  print(f"Ratio de Solvabilité: {ratio_solvabilite}")                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Chiffre d'Affaires: # Rapport Financier Annuel 2025

**Entreprise :** NovaTech Solutions SAS

**Secteur :** Technologies de l'information et services numériques

**Exercice :** 1er janvier 2025 – 31 décembre 2025

**Siège social :** Paris, France


Document fictif à usage pédagogique


## 1. Présentation Générale de l’Entreprise

NovaTech Solutions SAS est une entreprise fictive spécialisée dans le développement de
solutions logicielles pour les entreprises de taille moyenne et les grandes organisations.
En 2025, l’entreprise a poursuivi sa stratégie de croissance basée sur l’innovation, la
qualité de service et l’expansion de son portefeuille clients.

L’année 2025 a été marquée par une forte demande en solutions cloud, en intelligence
artificielle et en cybersécurité. NovaTech Solutions a su tirer parti de ces tendances pour
renforcer sa position sur le marché.


## 2. Compte de Résultat Simplifié
---
## 2. Compte de Résultat Simplifié


|Poste|Montant (€)|
|---|---|
|Chiffre d’affaires|5 200 000|
|Coûts d’exploitation|-3 100 000|
|Marge brute|2 100 000|
|Charges de personnel|-1 200 000|
|Autres charges|-450 000|
|Résultat d’exploitation|450 000|
|Impôts|-120 000|
|Résultat net|330 000|


## 3. Bilan Simplifié au 31 Décembre 2025


|Actif|Montant (€)|Passif|Montant (€)|
|---|---|---|---|
|Immobilisations|1 800 000|Capitaux propres|2 300 000|
|Actif circulant|1 400 000|Dettes long terme|500 000|
|Trésorerie|600 000|Dettes court terme|1 000 000|
|Total Actif|3 800 000|Total Passif|3 800 000|


## 4. Analyse Financière et Perspectives

Les résultats financiers de NovaTech Solutions SAS pour l’exercice 2025 montrent une
situation saine et équilibrée. La croissance du chiffre d’affaires (+12 % par rapport à 2024)
reflète la pertinence des choix stratégiques opérés.
---
La structure financière de l’entreprise reste solide, avec un niveau de trésorerie
confortable permettant de financer les investissements futurs sans recourir excessivement
à l’endettement.

Pour l’année 2026, NovaTech Solutions prévoit d’investir dans la recherche et
développement, notamment dans les domaines de l’intelligence artificielle et de
l’automatisation des processus métiers, tout en poursuivant son expansion à
l’international.
Résultat Net: ## 2. Compte de Résultat Simplifié


|Poste|Montant (€)|
|---|---|
|Chiffre d’affaires|5 200 000|
|Coûts d’exploitation|-3 100 000|
|Marge brute|2 100 000|
|Charges de personnel|-1 200 000|
|Autres charges|-450 000|
|Résultat d’exploitation|450 000|
|Impôts|-120 000|
|Résultat net|330 000|


## 3. Bilan Simplifié au 31 Décembre 2025


|Actif|Montant (€)|Passif|Montant (€)|
|---|---|---|---|
|Immobilisations|1 800 000|Capitaux propres|2 300 000|
|Actif circulant|1 400 000|Dettes long terme|500 000|
|Trésorerie|600 000|Dettes court terme|1 000 000|
|Total Actif|3 800 000|Total Passif|3 800 000|


## 4. Analyse Financière et Perspectives

Les résultats financiers de NovaTech Solutions SAS pour l’exercice 2025 montrent une
situation saine et équilibrée. La croissance du chiffre d’affaires (+12 % par rapport à 2024)
reflète la pertinence des choix stratégiques opérés.
---
# Rapport Financier Annuel 2025

**Entreprise :** NovaTech Solutions SAS

**Secteur :** Technologies de l'information et services numériques

**Exercice :** 1er janvier 2025 – 31 décembre 2025

**Siège social :** Paris, France


Document fictif à usage pédagogique


## 1. Présentation Générale de l’Entreprise

NovaTech Solutions SAS est une entreprise fictive spécialisée dans le développement de
solutions logicielles pour les entreprises de taille moyenne et les grandes organisations.
En 2025, l’entreprise a poursuivi sa stratégie de croissance basée sur l’innovation, la
qualité de service et l’expansion de son portefeuille clients.

L’année 2025 a été marquée par une forte demande en solutions cloud, en intelligence
artificielle et en cybersécurité. NovaTech Solutions a su tirer parti de ces tendances pour
renforcer sa position sur le mar

...

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Create headers                                                                                                 
  print(chirurgien_excel(nom_fichier='Audit_TechNova.xlsx', cellule='A1', valeur='Indicateur', action='ECRIRE'))   
  print(chirurgien_excel(nom_fichier='Audit_TechNova.xlsx', cellule='B1', valeur='Valeur', action='ECRIRE'))       
                                                                                                                   
  # Apply formatting to header row                                                                                 
  print(chirurgien_excel(nom_fichier='Audit_TechNova.xlsx', cellule='A1:B1', valeur='', action='TITRE_STYLYSE'))   
  print(chirurgien_excel(nom_fichier='Audit_TechNova.xlsx', cellule='A1', valeur='', action='GRAS'))               
  print(chirurgien_excel(nom_fichier='Audit_TechNova.xlsx', cellule='A1', valeur='', action='ROUGE'))              
  print(chirurgien_excel(nom_fichier='Audit_TechNova.xlsx', cellule='B1', valeur='', action='GRAS'))               
  print(chirurgien_excel(nom_fichier='Audit_TechNova.xlsx', cellule='B1', valeur='', action='ROUGE'))              
                                                                                                                   
                                                                                                                   
  # Populate data                                                                                                  
  print(chirurgien_excel(nom_fichier='Audit_TechNova.xlsx', cellule='A2', valeur='Chiffre d\'Affaires',            
  action='ECRIRE'))                                                                                                
  print(chirurgien_excel(nom_fichier='Audit_TechNova.xlsx', cellule='B2', valeur=chiffre_affaires_val,             
  action='ECRIRE'))                                                                                                
                                                                                                                   
  print(chirurgien_excel(nom_fichier='Audit_TechNova.xlsx', cellule='A3', valeur='Résultat Net',                   
  action='ECRIRE'))                                                                                                
  print(chirurgien_excel(nom_fichier='Audit_TechNova.xlsx', cellule='B3', valeur=resultat_net_val,                 
  action='ECRIRE'))                                                                                                
                                     

Execution logs:
Succès Excel : ECRIRE sur A1.
Succès Excel : ECRIRE sur B1.
Succès Excel : TITRE_STYLYSE sur A1:B1.
Succès Excel : GRAS sur A1.
Succès Excel : ROUGE sur A1.
Succès Excel : GRAS sur B1.
Succès Excel : ROUGE sur B1.
Succès Excel : ECRIRE sur A2.
Succès Excel : ECRIRE sur B2.
Succès Excel : ECRIRE sur A3.
Succès Excel : ECRIRE sur B3.
Succès Excel : ECRIRE sur A4.
Succès Excel : ECRIRE sur B4.
Succès Excel : ECRIRE sur A5.
Succès Excel : ECRIRE sur B5.

Final answer: L'analyse du document PDF est terminée et le fichier 'Audit_TechNova.xlsx' a été créé avec les 
informations extraites.

[Step 4: Duration 4.90 seconds| Input tokens: 28,653 | Output tokens: 4,000]


💡 RÉSULTAT :
L'analyse du document PDF est terminée et le fichier 'Audit_TechNova.xlsx' a été créé avec les informations extraites.


In [ ]:
# --- TEST DIAGNOSTIC DE LA CLÉ ---
import google.generativeai as genai
import os

# Ta clé (elle est déjà en mémoire si tu as lancé les cellules précédentes)
if 'CLE_GOOGLE_VISION' in globals():
    genai.configure(api_key=CLE_GOOGLE_VISION)
    print(f"🔑 Clé détectée : {CLE_GOOGLE_VISION[:5]}...*****")
else:
    print("⚠️ Remets ta clé dans la variable CLE_GOOGLE_VISION")

try:
    # On essaie le modèle le plus récent pour voir si la clé répond
    print("📡 Tentative de contact direct avec Google (sans LangChain)...")

    # Test avec le modèle 'text-embedding-004' (le plus récent)
    result = genai.embed_content(
        model="models/text-embedding-004",
        content="Test de connexion",
        task_type="retrieval_document"
    )

    print("\n✅ SUCCÈS ! Ta clé fonctionne parfaitement.")
    print(f"   Google a renvoyé un vecteur de {len(result['embedding'])} chiffres.")
    print("   Conclusion : Le problème vient de LangChain, pas de ta clé.")

except Exception as e:
    print(f"\n❌ ÉCHEC. Message d'erreur : {e}")
    if "403" in str(e):
        print("   -> Là oui, c'est un problème de clé ou de permission.")
    elif "404" in str(e):
        print("   -> Le modèle demandé n'est pas dispo sur ton compte/région.")

🔑 Clé détectée : AIzaS...*****
📡 Tentative de contact direct avec Google (sans LangChain)...

❌ ÉCHEC. Message d'erreur : 404 POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:embedContent?%24alt=json%3Benum-encoding%3Dint: models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.
   -> Le modèle demandé n'est pas dispo sur ton compte/région.


In [ ]:
# --- DÉTECTIVE DES MODÈLES DISPONIBLES ---
import google.generativeai as genai
import os

# Ta clé (déjà en mémoire normalement)
if 'CLE_GOOGLE_VISION' in globals():
    genai.configure(api_key=CLE_GOOGLE_VISION)
else:
    print("⚠️ Clé introuvable, remets-la.")

print("🔍 Recherche des modèles d'Embedding disponibles pour ta clé...")

try:
    # On demande la liste de tous les modèles
    compteur = 0
    for m in genai.list_models():
        # On cherche ceux qui savent faire de l'embedding (embedContent)
        if 'embedContent' in m.supported_generation_methods:
            print(f"✅ TROUVÉ : {m.name}")
            compteur += 1

    if compteur == 0:
        print("❌ Aucun modèle d'embedding trouvé. C'est peut-être un souci de région (Europe).")
    else:
        print(f"\n👉 Copie le nom exact d'un des modèles ci-dessus (ex: models/embedding-001).")

except Exception as e:
    print(f"Erreur fatale : {e}")

🔍 Recherche des modèles d'Embedding disponibles pour ta clé...
✅ TROUVÉ : models/gemini-embedding-001

👉 Copie le nom exact d'un des modèles ci-dessus (ex: models/embedding-001).
